# F26-13 — Phase 2: Data Acquisition & Setup

**Confidence-Aware Self-Prompting for Robust Polyp Segmentation Under Image and Domain Shifts**
AI2002 Artificial Intelligence (BDS-5A), FAST-NUCES Lahore, Fall 2026 — Track A

| Member | ID |
|---|---|
| Kashish Fatima (lead) | 24L-2605 |
| Khushbakht Sohail | 24L-2604 |
| Zahra Saeed | 24L-2512 |

Repository: https://github.com/kashishfatima999/Confidence-Aware-Self-Prompting-for-Robust-Polyp-Segmentation

This notebook reproduces every Phase 2 step from a clean Colab session (T4 GPU, *Runtime → Run all*, about 10 minutes):

1. environment and tests
2. Kvasir-SEG: download, verification, mask cleaning, seeded split, YOLO labels
3. CVC-ClinicDB: download, verification, cleaning, YOLO labels (external test set only)
4. exploratory statistics of both datasets
5. the image-shift test set (corrupted copies of the Kvasir test split)
6. a smoke test of the baseline pipeline (YOLO → SAM 2) and an overlay check of the labels

Every number printed here is the one quoted in `docs/data_report.md`; the reasoning behind each step is in `docs/decision_log.md` (entries `DL-xx` referenced below). The code lives in `src/data/`; this notebook only calls it.

## 1. GPU, Drive and code

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Our repository (main branch) and the baseline repository (read-only, git-ignored, no license: DL-16).
# rm -rf makes the cell safe to re-run after a session restart.
!rm -rf /content/proj
!git clone https://github.com/kashishfatima999/Confidence-Aware-Self-Prompting-for-Robust-Polyp-Segmentation.git /content/proj
%cd /content/proj
!git clone https://github.com/sajjad-sh33/YOLO_SAM2.git external/YOLO_SAM2
!git log -1 --format="Our commit: %h %s"
!git -C external/YOLO_SAM2 log -1 --format="Baseline commit: %H (%cd)"

## 2. Environment and tests (DL-13, DL-22, DL-26)

On Colab the preinstalled numpy, Pillow, OpenCV, torch and torchvision are used as they are; only the model packages pinned in `requirements-colab.txt` are installed (`requirements.txt` is for the local CPU environment). The unit and pipeline tests need no dataset: they build small synthetic datasets and run all three preparation scripts on them.

In [ ]:
# Only the model packages are installed here. requirements.txt (numpy/Pillow/OpenCV pins) is for the local CPU
# environment: Colab already ships those packages, and upgrading Pillow inside a running Colab kernel breaks
# every later Pillow import (DL-26, challenge C7).
!pip install -q -r requirements-colab.txt
!SAM2_BUILD_CUDA=0 pip install -q "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"
!pip freeze | grep -i -E "^torch==|^torchvision==|^ultralytics==|^numpy==|^pillow==|^opencv|SAM-2"

In [ ]:
!python -m unittest discover -s tests -v 2>&1 | tail -8

## 3. Model weights (DL-12, DL-17)

Stored in Google Drive so they are downloaded once. The baseline's shipped Kvasir YOLO weights are used for the smoke test only, never for a reported result (DL-17).

In [ ]:
# Weights live in Google Drive so they are downloaded only once (-nc skips existing files)
W = "/content/drive/MyDrive/F26-13/weights"
!mkdir -p {W}
!grep -i large external/YOLO_SAM2/checkpoints/download_ckpts.sh
!wget -q -nc -P {W} https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
!cd {W} && python -c "from ultralytics import YOLO; YOLO('yolov8m.pt')"
!ls -la {W}

## 4. Kvasir-SEG — primary dataset (DL-01, DL-02, DL-04 to DL-11, DL-14)

Source: Simula Research Laboratory, https://datasets.simula.no/kvasir-seg/ (Jha et al., 2020). Member 1 downloaded the official zip locally (46,227,172 bytes, SHA-256 `03b30e21…3478f7`, 2026-10-07). The official link returned 0 bytes from Colab on 2026-10-09, so here the author's Kaggle copy `debeshjha1/kvasirseg` is used; it stores boxes as one CSV per image, so the cell rebuilds `kavsir_bboxes.json` in the official format first (DL-26). The YOLO labels do not depend on that file: they are derived from the masks (DL-06).

In [ ]:
import os, glob, json, shutil
import kagglehub
import pandas as pd
from PIL import Image
os.chdir("/content/proj")

p = kagglehub.dataset_download("debeshjha1/kvasirseg")
src = f"{p}/Kvasir-SEG/Kvasir-SEG"
dest = "data/raw/kvasir-seg/Kvasir-SEG"

shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(f"{src}/images", f"{dest}/images")   # copy, never move
shutil.copytree(f"{src}/masks", f"{dest}/masks")

boxes = {}
for csv_path in sorted(glob.glob(f"{src}/bbox/*.csv")):
    image_id = os.path.splitext(os.path.basename(csv_path))[0]
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    w, h = Image.open(f"{dest}/images/{image_id}.jpg").size
    boxes[image_id] = {"height": h, "width": w,
                       "bbox": [{"label": "polyp", "xmin": int(r["xmin"]), "ymin": int(r["ymin"]),
                                 "xmax": int(r["xmax"]), "ymax": int(r["ymax"])} for _, r in df.iterrows()]}
with open(f"{dest}/kavsir_bboxes.json", "w") as f:
    json.dump(boxes, f)
print("Box entries written:", len(boxes))
!ls {dest}; ls {dest}/images | wc -l; ls {dest}/masks | wc -l

`prepare_kvasir.py` verifies the raw data (counts, pairing, sizes, full decode, empty masks), binarises the masks at 127 to PNG, derives one box per polyp region (specks < 0.05% of the image ignored), reuses the committed 700/100/200 split (seed 42), writes YOLO labels and two reports.

In [ ]:
!python src/data/prepare_kvasir.py

**EDA insight 1 — the official box file is unreliable (DL-06).** For 18 of 1,000 images the official boxes disagree with the masks: in 10 images the official box starts at (0, 0) and covers 82–100% of the frame while the polyp covers 46–90%; in 8 images there is an extra box around a compression speck. This is why the labels are derived from the masks.

In [ ]:
import pandas as pd
cmp = pd.read_csv("data/processed/kvasir/reports/box_comparison.csv")
print(f"images: {len(cmp)}   agree: {int(cmp.agrees.sum())}   mean extent IoU: {cmp.extent_iou.mean():.4f}")
print("\nDisagreeing images (official vs mask-derived boxes):")
print(cmp[~cmp.agrees].sort_values("extent_iou").to_string(index=False))

## 5. CVC-ClinicDB — external test set (DL-03, DL-24, DL-28)

Source: Bernal et al. (2015), https://polyp.grand-challenge.org/CVCClinicDB/; downloaded from the Kaggle mirror `balraj98/cvcclinicdb` (PNG version), the mirror the baseline uses. Research and educational use only. All 612 frames go into a single `test` split: this dataset is never used for training or threshold selection.

In [ ]:
p2 = kagglehub.dataset_download("balraj98/cvcclinicdb")
shutil.rmtree("data/raw/cvc-clinicdb", ignore_errors=True)
shutil.copytree(f"{p2}/PNG", "data/raw/cvc-clinicdb/PNG")   # copy, never move
!ls "data/raw/cvc-clinicdb/PNG/Original" | wc -l
!python src/data/prepare_clinicdb.py

## 6. Exploratory statistics of both datasets (DL-25)

Per-image polyp area, connected polyp regions, box aspect ratio, brightness (mean grey value) and contrast (standard deviation of grey values), computed with the same code for both datasets. Figures go to `docs/figures/stats_*.png`, tables to `docs/tables/`.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from the repository root. Figures go to docs/figures/, tables to docs/tables/ (DL-27).
ROOT = Path.cwd()
FIG = ROOT / "docs" / "figures"
TAB = ROOT / "docs" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

def find_image(img_dir, mask_path):
    for ext in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
        candidate = img_dir / (mask_path.stem + ext)
        if candidate.exists():
            return candidate
    return None

def analyse_dataset(name, base, splits):
    rows = []

    for split in splits:
        img_dir = base / "images" / split
        mask_dir = base / "masks" / split

        if not mask_dir.exists() or not img_dir.exists():
            print(f"Skipping missing folders: {name}/{split}")
            continue

        for mask_path in sorted(mask_dir.iterdir()):
            if mask_path.suffix.lower() not in [
                ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
            ]:
                continue

            img_path = find_image(img_dir, mask_path)
            if img_path is None:
                continue

            img = cv2.imread(str(img_path))
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

            if img is None or mask is None:
                continue

            h, w = mask.shape
            binary = (mask > 127).astype(np.uint8)
            area_percent = binary.mean() * 100

            count, _, stats, _ = cv2.connectedComponentsWithStats(binary)
            min_area = max(1, int(binary.size * 0.0005))
            boxes = []

            for i in range(1, count):
                x, y, bw, bh, component_area = stats[i]
                if component_area >= min_area and bh > 0:
                    boxes.append(bw / bh)

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            rows.append({
                "dataset": name,
                "split": split,
                "image": img_path.name,
                "width": w,
                "height": h,
                "polyp_area_percent": area_percent,
                "polyps_per_image": len(boxes),
                "box_aspect_ratio_mean": (
                    float(np.mean(boxes)) if boxes else np.nan
                ),
                "brightness": float(gray.mean()),
                "contrast": float(gray.std())
            })

    return pd.DataFrame(rows)

kvasir = analyse_dataset(
    "Kvasir-SEG",
    ROOT / "data" / "processed" / "kvasir",
    ["train", "val", "test"]
)

clinic = analyse_dataset(
    "CVC-ClinicDB",
    ROOT / "data" / "processed" / "cvc-clinicdb",
    ["test"]
)

df = pd.concat([kvasir, clinic], ignore_index=True)

if df.empty:
    raise ValueError("No images were analysed. Check the dataset folders.")

df.to_csv(TAB / "dataset_statistics.csv", index=False)

summary = df.groupby("dataset").agg(
    images=("image", "count"),
    mean_width=("width", "mean"),
    mean_height=("height", "mean"),
    mean_polyp_area_percent=("polyp_area_percent", "mean"),
    mean_polyps_per_image=("polyps_per_image", "mean"),
    mean_box_aspect_ratio=("box_aspect_ratio_mean", "mean"),
    mean_brightness=("brightness", "mean"),
    mean_contrast=("contrast", "mean")
).round(3)

print("DATASET COMPARISON")
display(summary)
summary.to_csv(TAB / "dataset_comparison.csv")

metrics = [
    ("polyp_area_percent", "Polyp area (%)"),
    ("polyps_per_image", "Connected polyp regions per image"),
    ("box_aspect_ratio_mean", "Bounding-box aspect ratio"),
    ("brightness", "Mean brightness"),
    ("contrast", "Grayscale contrast")
]

for column, title in metrics:
    plot_data = df.dropna(subset=[column])
    if plot_data.empty:
        continue

    plot_data.boxplot(column=column, by="dataset")
    plt.title(title)
    plt.suptitle("")
    plt.xlabel("Dataset")
    plt.ylabel(title)
    plt.tight_layout()
    plt.savefig(FIG / f"stats_{column}.png", dpi=150)
    plt.show()
    plt.close()

print(f"Kvasir images analysed: {len(kvasir)}")
print(f"ClinicDB images analysed: {len(clinic)}")
print(f"Figures saved in: {FIG.resolve()}")
print(f"Tables saved in: {TAB.resolve()}")

**EDA insight 2 — there is a measurable domain shift.** CVC-ClinicDB polyps are smaller on average (9.2% of the frame vs 15.4%), its frames are darker (mean brightness 76 vs 97) and lower-contrast (54 vs 59), and all its frames are 384 × 288 while Kvasir-SEG has 333 distinct resolutions. This is the shift the cross-dataset evaluation in later phases will test. The statistics describe the datasets; they do not by themselves predict the size of the performance drop.

## 7. Image-shift test set (DL-20)

Blur, brightness/contrast and Gaussian noise at three strengths each, applied to the **200 test images only**; masks and labels are copied unchanged. Noise is seeded per image, so reruns are identical.

In [ ]:
!python src/data/make_corruptions.py

In [ ]:
from pathlib import Path
import cv2, glob, matplotlib.pyplot as plt
ids = Path("splits/kvasir_test.txt").read_text().split()
i = ids[0]
settings = sorted(glob.glob("data/processed/kvasir-corrupted/*_s*"))
fig, axes = plt.subplots(1, 10, figsize=(25, 3))
axes[0].imshow(cv2.cvtColor(cv2.imread(f"data/processed/kvasir/images/test/{i}.jpg"), cv2.COLOR_BGR2RGB))
axes[0].set_title("original")
for ax, s in zip(axes[1:], settings):
    ax.imshow(cv2.cvtColor(cv2.imread(f"{s}/images/test/{i}.png"), cv2.COLOR_BGR2RGB))
    ax.set_title(Path(s).name, fontsize=8)
for ax in axes: ax.axis("off")
plt.savefig("corruption_examples.png", dpi=100, bbox_inches="tight"); plt.show()

## 8. Smoke test: YOLO → SAM 2 on 5 test images (DL-17, DL-18)

Shows that the processed data is ready for model implementation: the baseline pipeline runs end to end on it. YOLO is run at confidence 0.05 so every candidate box is printed; only boxes ≥ 0.5 are passed to SAM 2, as in the baseline's `Test.py`. On 2026-10-09 this gave Dice 0.96–0.97 on 4 images and 0.00 on 1 image whose polyp YOLO did not detect at all: the detector, not the segmenter, is the weak point, which is what this project targets. Not a reported result (shipped weights, DL-17).

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

W = Path("/content/drive/MyDrive/F26-13/weights")
yolo = YOLO("external/YOLO_SAM2/YOLO_Checkpoints/Kvasir_yolov8m.pt")   # SMOKE TEST ONLY (DL-17)
sam = SAM2ImagePredictor(build_sam2("configs/sam2/sam2_hiera_l.yaml", str(W / "sam2_hiera_large.pt"), device="cuda"))

ids = Path("splits/kvasir_test.txt").read_text().split()[:5]
fig, axes = plt.subplots(5, 3, figsize=(10, 16))
for row, image_id in enumerate(ids):
    img_path = f"data/processed/kvasir/images/test/{image_id}.jpg"
    rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(f"data/processed/kvasir/masks/test/{image_id}.png", 0) > 127

    det = yolo.predict(img_path, imgsz=640, conf=0.05, verbose=False)[0]
    boxes_xyxy, confs = det.boxes.xyxy.cpu().numpy(), det.boxes.conf.cpu().numpy()

    sam.set_image(rgb)
    pred = np.zeros(gt.shape, bool)
    print(f"\n{image_id}")
    for b, c in zip(boxes_xyxy, confs):
        kept = c >= 0.5
        print(f"  box={np.round(b).astype(int).tolist()}  conf={c:.3f}  {'KEPT' if kept else 'dropped by baseline'}")
        if kept:
            m, _, _ = sam.predict(box=b, multimask_output=False)
            pred |= m[0] > 0
    dice = 2 * (pred & gt).sum() / (pred.sum() + gt.sum() + 1e-8)
    print(f"  Dice = {dice:.3f}")

    axes[row, 0].imshow(rgb)
    for b, c in zip(boxes_xyxy, confs):
        axes[row, 0].add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, lw=2,
                                             color="lime" if c >= .5 else "red"))
    axes[row, 1].imshow(gt, cmap="gray"); axes[row, 2].imshow(pred, cmap="gray")
for ax, t in zip(axes[0], ["image + YOLO boxes (green kept, red dropped)", "ground-truth mask", "SAM 2 mask"]):
    ax.set_title(t, fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.savefig("smoke_test.png", dpi=100); plt.show()

## 9. Overlay check of the labels (task 3.5)

20 random test images per dataset: green = mask outline, red = YOLO label box. Every red box should wrap its green outline. On 2026-10-09: 20/20 correct on both datasets.

In [ ]:
import random
def overlay(dataset, ext, out_name):
    folder = Path(f"data/processed/{dataset}")
    all_ids = sorted(q.stem for q in (folder / "images/test").glob(f"*.{ext}"))
    pick = random.Random(0).sample(all_ids, 20)
    fig, axes = plt.subplots(4, 5, figsize=(18, 13))
    for ax, i in zip(axes.flat, pick):
        rgb = cv2.cvtColor(cv2.imread(str(folder / f"images/test/{i}.{ext}")), cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        mask = cv2.imread(str(folder / f"masks/test/{i}.png"), 0) > 127
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(rgb, contours, -1, (0, 255, 0), 2)
        for line in (folder / f"labels/test/{i}.txt").read_text().split("\n"):
            if not line.strip(): continue
            _, xc, yc, bw, bh = map(float, line.split())
            cv2.rectangle(rgb, (int((xc-bw/2)*w), int((yc-bh/2)*h)), (int((xc+bw/2)*w), int((yc+bh/2)*h)), (255, 0, 0), 2)
        ax.imshow(rgb); ax.set_title(i[:14], fontsize=8); ax.axis("off")
    plt.suptitle(f"{dataset}: green = polyp outline, red = YOLO label box"); plt.tight_layout()
    plt.savefig(out_name, dpi=90); plt.show()

overlay("kvasir", "jpg", "overlay_kvasir.png")
overlay("cvc-clinicdb", "png", "overlay_clinicdb.png")

## 10. Final dataset summary

In [ ]:
import json
kv = json.load(open("data/processed/kvasir/reports/verification.json"))
cl = json.load(open("data/processed/cvc-clinicdb/reports/verification.json"))
man = json.load(open("data/processed/kvasir-corrupted/manifest.json"))
rows = [
    ["Role", "train / val / test", "external test only", "image-shift test only"],
    ["Images", " / ".join(str(kv["splits"][s]["images"]) for s in ("train", "val", "test")), cl["images"],
     f'{len(man["settings"])} settings x {man["images_per_setting"]} = {len(man["settings"]) * man["images_per_setting"]}'],
    ["Mask-derived boxes", " / ".join(str(kv["splits"][s]["boxes"]) for s in ("train", "val", "test")), cl["total_boxes"], f'{kv["splits"]["test"]["boxes"]} per setting (copied)'],
    ["Resolution", f'{kv["distinct_resolutions"]} distinct, {kv["width_min"]}-{kv["width_max"]} x {kv["height_min"]}-{kv["height_max"]}', cl["image_resolution"], "as Kvasir test"],
    ["Unreadable files / empty masks", f'{kv["unreadable_files"]} / {kv["empty_masks"]}', f'{cl["unreadable_files"]} / {cl["empty_masks"]}', "-"],
    ["Mean polyp area (%)", " / ".join(str(kv["splits"][s]["polyp_area_percent_mean"]) for s in ("train", "val", "test")), cl["mean_polyp_area_percent"], "as Kvasir test"],
]
print(pd.DataFrame(rows, columns=["", "Kvasir-SEG", "CVC-ClinicDB", "Kvasir-SEG corrupted"]).to_string(index=False))

## 11. Save figures, tables and reports to Google Drive

Everything in `/content` disappears when the session ends; these copies in `MyDrive/F26-13/` are what gets committed back to the repository.

In [ ]:
!mkdir -p /content/drive/MyDrive/F26-13/figures /content/drive/MyDrive/F26-13/tables /content/drive/MyDrive/F26-13/reports docs/figures
!cp smoke_test.png corruption_examples.png overlay_kvasir.png overlay_clinicdb.png docs/figures/
!cp docs/figures/*.png /content/drive/MyDrive/F26-13/figures/
!cp docs/tables/*.csv /content/drive/MyDrive/F26-13/tables/
!cp data/processed/kvasir/reports/*.json data/processed/kvasir/reports/*.csv data/processed/cvc-clinicdb/reports/*.json data/processed/kvasir-corrupted/manifest.json /content/drive/MyDrive/F26-13/reports/
!ls -la /content/drive/MyDrive/F26-13/figures /content/drive/MyDrive/F26-13/tables /content/drive/MyDrive/F26-13/reports

## References

1. M. Mansoori et al. *Self-Prompting Polyp Segmentation in Colonoscopy using Hybrid YOLO-SAM 2 Model.* arXiv:2409.09484, 2024. Code: https://github.com/sajjad-sh33/YOLO_SAM2
2. N. Ravi et al. *SAM 2: Segment Anything in Images and Videos.* arXiv:2408.00714, 2024.
3. D. Jha et al. *Kvasir-SEG: A Segmented Polyp Dataset.* MMM 2020. arXiv:1911.07069.
4. J. Bernal et al. *WM-DOVA maps for accurate polyp highlighting in colonoscopy.* Computerized Medical Imaging and Graphics 43:99–111, 2015.
5. G. Jocher et al. *Ultralytics YOLOv8.* https://github.com/ultralytics/ultralytics